### Train model nhận dạng ngôn ngữ ký hiệu

## 1. Import Thư Viện

In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Load Dataset

In [9]:
from src.dataset import SignLanguageDataset
from src.m2 import SignLanguageModel

# Define paths
DATA_DIR = "./data/processed_frames/asl_alphabet_train/asl_alphabet_train"  # Path to training data
BATCH_SIZE = 32
NUM_WORKERS = 4

# Data preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Load dataset
print("Loading dataset...")
dataset = SignLanguageDataset(root_dir=DATA_DIR, transform=transform)
print(f"Total images: {len(dataset)}")
print(f"Number of classes: {dataset.get_num_classes()}")

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                         shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                        shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Loading dataset...
Total images: 87000
Number of classes: 29
Train samples: 69600
Validation samples: 17400


## 3. Xây Dựng Model

In [10]:
# Initialize model
print("Initializing model...")
num_classes = dataset.get_num_classes()
model = SignLanguageModel(num_classes=num_classes)
model = model.to(device)

# Print model architecture
print(f"\nModel loaded with {num_classes} classes")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Initializing model...


c:\Users\Laptop 3H\miniconda3\envs\ipdlm\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Laptop 3H\miniconda3\envs\ipdlm\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Laptop 3H/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:19<00:00, 2.38MB/s]



Model loaded with 29 classes
Model parameters: 11,191,389


## 4. Cấu Hình Training

In [11]:
# Training hyperparameters
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {device}")

Training configuration:
  Epochs: 20
  Learning rate: 0.001
  Batch size: 32
  Device: cuda


c:\Users\Laptop 3H\miniconda3\envs\ipdlm\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


## 5. Vòng Lặp Training

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss / (total / BATCH_SIZE), 
                         'acc': correct / total})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validating')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': running_loss / (total / BATCH_SIZE), 
                             'acc': correct / total})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


# Training loop
print("Starting training...\n")
best_val_acc = 0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Learning rate scheduler
    scheduler.step(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), './models/best_model_m2.pth')
        print(f"✓ Best model saved (Val Acc: {val_acc:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("\n" + "="*50)
print("Training completed!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Starting training...


Epoch 1/20
--------------------------------------------------


Validating: 100%|██████████| 544/544 [01:04<00:00,  8.44it/s, loss=0.0755, acc=0.976]


Train Loss: 0.0868 | Train Acc: 0.9753
Val Loss: 0.0755 | Val Acc: 0.9758


RuntimeError: Parent directory ../models does not exist.

## 6. Đánh Giá Hiệu Xuất

In [ ]:
# Load best model
model.load_state_dict(torch.load('../models/best_model_m2.pth'))
model.eval()

# Get predictions on validation set
all_predictions = []
all_labels = []
all_probabilities = []

print("Generating predictions on validation set...")
with torch.no_grad():
    for images, labels in tqdm(val_loader):
        images = images.to(device)
        outputs = model(images)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        
        _, predicted = torch.max(outputs.data, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_probabilities = np.array(all_probabilities)

# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Detailed classification report
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT")
print("="*50)
class_names = [dataset.get_label_name(i) for i in range(dataset.get_num_classes())]
print(classification_report(all_labels, all_predictions, target_names=class_names))

: 

: 

## 7. Biểu Đồ & Trực Quan Hóa

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Training and Validation Loss
axes[0, 0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Validation Loss', marker='s', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. Training and Validation Accuracy
axes[0, 1].plot(history['train_acc'], label='Train Accuracy', marker='o', linewidth=2)
axes[0, 1].plot(history['val_acc'], label='Validation Accuracy', marker='s', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy', fontsize=12)
axes[0, 1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], 
            xticklabels=class_names[:10], yticklabels=class_names[:10],
            cbar_kws={'label': 'Count'})
axes[1, 0].set_xlabel('Predicted Label', fontsize=12)
axes[1, 0].set_ylabel('True Label', fontsize=12)
axes[1, 0].set_title('Confusion Matrix (first 10 classes)', fontsize=14, fontweight='bold')

# 4. Metrics Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = axes[1, 1].bar(metrics, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('Score', fontsize=12)
axes[1, 1].set_title('Performance Metrics', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{value:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/training_results_m2.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Training results saved to '../outputs/training_results_m2.png'")

: 

: 

## 8. Lưu Mô Hình & Tóm Tắt

In [ ]:
# Save training summary
summary = {
    'Model': 'ResNet18 (Pretrained)',
    'Dataset Size': len(dataset),
    'Train/Val Split': f"{len(train_dataset)}/{len(val_dataset)}",
    'Batch Size': BATCH_SIZE,
    'Learning Rate': LEARNING_RATE,
    'Num Epochs': len(history['train_loss']),
    'Number of Classes': num_classes,
    'Accuracy': f"{accuracy:.4f}",
    'Precision': f"{precision:.4f}",
    'Recall': f"{recall:.4f}",
    'F1-Score': f"{f1:.4f}",
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('../outputs/training_summary_m2.csv', index=False)
print("\n✓ Summary saved to '../outputs/training_summary_m2.csv'")

# Save detailed metrics
metrics_df = pd.DataFrame({
    'Epoch': range(1, len(history['train_loss']) + 1),
    'Train Loss': history['train_loss'],
    'Train Accuracy': history['train_acc'],
    'Val Loss': history['val_loss'],
    'Val Accuracy': history['val_acc']
})
metrics_df.to_csv('../outputs/training_metrics_m2.csv', index=False)
print("✓ Detailed metrics saved to '../outputs/training_metrics_m2.csv'")

print("\n" + "="*50)
print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
print("="*50)

: 

: 